# ONNM - Colab training

Runs the **same code** as a local run: this notebook unpacks the repo, installs it, and
calls the same `scripts/` entry points. A result produced here is comparable to one
produced on the local RX 7900 XT, which is the whole point - the job is to explain a
regression, and that needs two runs that differ in exactly one thing.

## What this run is for

The overnight run scored **0.8629** macro ROC-AUC against the full run's **0.8905**.
ROC-AUC is threshold-independent, so that is a real loss of ranking quality, not something
a threshold can recover. Two things changed at once - aggressive augmentation and OHEM -
so neither can be blamed yet. Cells 9 and 10 run them separately.

## Before you start

1. `Runtime -> Change runtime type -> T4 GPU` (or better, if you have Pro).
2. This notebook expects the following in your Drive at
   **`MyDrive/OSTEONEURALNETWORK/`**:

   | file | what it is |
   |---|---|
   | `onnm-code.zip` | the repo source |
   | `BTXRD.zip` | the 874 MB dataset |
   | `splits.json` | the exact split used locally, for comparability |

3. Run the cells in order. Cell 8 is a 2-epoch smoke run - **do not skip it**, it proves
   the whole path works before an hour is spent on a real run.

**Free-tier caveat:** Colab disconnects on idle and caps sessions at ~12 h. The configs
below run 40 epochs with early stopping (patience 15), which lands well inside that. Do
not try to run `overnight.yaml`'s nominal 150 epochs here.


In [ ]:
# --- Cell 1: what hardware did we actually get? -----------------------------
# Worth knowing before anything else. The free tier is a T4 (Turing, sm_75),
# which has NO bfloat16 - the project trains in bf16 locally, and the training
# loop needs a GradScaler on fp16 that bf16 does not use. resolve_amp_dtype
# handles the fallback, but seeing it here means no surprises an hour in.
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
print("torch          ", torch.__version__)
print("cuda available ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device         ", torch.cuda.get_device_name(0))
    print("capability     ", torch.cuda.get_device_capability(0))
    print("bf16 supported ", torch.cuda.is_bf16_supported(), "  <- False on a T4; expected")
else:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU")


In [ ]:
# --- Cell 2: mount Drive and check the three inputs are there ---------------
import os
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/OSTEONEURALNETWORK")
REPO = Path("/content/OsteoNeuralNetwork-Model")
DATA = Path("/content/data")

expected = ["onnm-code.zip", "BTXRD.zip", "splits.json"]
missing = [f for f in expected if not (DRIVE / f).exists()]
if missing:
    raise SystemExit(
        f"missing from {DRIVE}: {missing}\n"
        f"present: {sorted(p.name for p in DRIVE.iterdir()) if DRIVE.exists() else None}"
    )

for f in expected:
    size = (DRIVE / f).stat().st_size / 1024 ** 2
    print(f"  {f:16s} {size:9.1f} MB")


In [ ]:
# --- Cell 3: unpack code and data to local disk -----------------------------
# Unzipping onto Drive itself would be pathologically slow: BTXRD is ~3.7k small
# files and Drive is a network filesystem. Everything goes to /content, which is
# local SSD, and results are copied back at the end.
#
# The dataset is symlinked into the repo's default location rather than
# configured elsewhere, because verify_data.py and make_splits.py take no
# --profile flag and would otherwise look somewhere different from training.
import json
import shutil

if not REPO.exists():
    !unzip -q "{DRIVE}/onnm-code.zip" -d /content/
print("code   ", REPO, "->", len(list(REPO.rglob("*.py"))), "python files")

if not (DATA / "BTXRD").exists():
    DATA.mkdir(parents=True, exist_ok=True)
    !unzip -q "{DRIVE}/BTXRD.zip" -d "{DATA}"
n_images = len(list((DATA / "BTXRD" / "images").glob("*")))
print("images ", DATA / "BTXRD", "->", n_images, "files   (expect 3746)")

# Symlink into the repo's default data_root, and copy the split in.
(REPO / "data" / "raw").mkdir(parents=True, exist_ok=True)
(REPO / "data" / "interim").mkdir(parents=True, exist_ok=True)
link = REPO / "data" / "raw" / "BTXRD"
if not link.exists():
    os.symlink(DATA / "BTXRD", link)
shutil.copy(DRIVE / "splits.json", REPO / "data" / "interim" / "splits.json")

# The split is copied rather than regenerated so Colab and local runs are
# literally the same partition. Comparing scores across different splits would
# be meaningless, and make_splits.py reproducing the same seed is not the same
# guarantee as using the same file.
with open(REPO / "data/interim/splits.json") as handle:
    split = json.load(handle)
print("split   train/val/test =",
      len(split["train"]), "/", len(split["val"]), "/", len(split["test"]))
print("        content_hash =", split["content_hash"], " (local run: db908a9afdc5d085)")
assert split["content_hash"] == "db908a9afdc5d085", "split differs from the local run"


In [ ]:
# --- Cell 4: install the project layer only ---------------------------------
# Colab ships a working CUDA torch. Do NOT reinstall it: replacing it reliably
# breaks the preinstalled CUDA libraries and costs a runtime restart.
#
# --no-deps on the editable install is what enforces that. pyproject.toml does
# not list torch (deliberately), but several of its dependencies do, and pip
# resolving them would happily pull a CPU-only wheel over Colab's CUDA build.
# Everything else the project needs is either installed on the line above or
# already present in Colab. Cell 5 (verify_env) is what confirms that claim
# rather than assuming it - if --no-deps skipped something real, it fails there.
%cd /content/OsteoNeuralNetwork-Model
!pip install -q monai==1.5.2 pydicom openpyxl
!pip install -q -e . --no-deps
print("installed")


## Gates

The project has numbered gates that must pass before a result means anything. Gate 6
(`overfit_check`) has **never been run** against the current pipeline - it proves gradients
actually reach the backbone. Running it here clears a blocking item from `TODO.md`.


In [ ]:
# --- Cell 5: gates 1 and 2 - environment, then data -------------------------
!python scripts/verify_env.py
!python scripts/verify_data.py


In [ ]:
# --- Cell 6: gate 3 - the test suite ----------------------------------------
# Needs no dataset; catches an install that half-worked.
!pip install -q pytest
!python -m pytest -q


In [ ]:
# --- Cell 7: gate 6 - overfit a tiny batch ----------------------------------
# A model that cannot drive loss to ~0 on 32 stratified images has a broken
# gradient path, and every number produced after that is noise. ~2 minutes.
!python scripts/overfit_check.py --profile colab --samples 32 --steps 200


In [ ]:
# --- Cell 8: smoke run - 2 epochs, end to end -------------------------------
# Proves loaders, AMP, scheduler, checkpointing and metrics all work together on
# this machine before an hour is committed. Look for two lines in the output:
#   'AMP: float16 (GradScaler on)'   - the T4 fallback fired correctly
#   'cudnn_enabled: True'            - the ROCm miopen=false flag was ignored
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --profile colab --epochs 2 --tag colab-smoke


## Community batch — approvals into training

Pulls the submissions you approved in the review console
(`streamlit run review_app.py`) and folds them into this run. Skip the cell for a
pure-BTXRD run; it is a no-op when nothing has been approved.

One command does the whole thing:

```
python scripts/sync_community.py --store <drive>/community
```

It claims the approved rows out of Cloudflare, writes the images, and **rebuilds
`configs/controls_manifest.csv`** — which is already the value of
`paths.controls_manifest` in `base.yaml`, so there is nothing to edit afterwards.
`make_splits.py` and `train.py` below pick the rows up on their own.

### Why the store is on Drive

Export **claims** rows: the Worker stamps `batch_id` so the same example cannot
enter two generations of training, and a claim cannot be undone from the client.
Colab wipes `/content` on disconnect. A batch claimed onto the local disk and then
lost to a disconnect is gone for good — the rows still read as exported, and the
images no longer exist anywhere. So the store lives in Drive and the manifest
records absolute paths into it, which `build_records` handles.

That also makes the store cumulative: every generation's approvals accumulate in
one place, and a fresh runtime rebuilds the full manifest from it without touching
the network (`--rebuild-only`).

### What is not automated, and why

Nothing pushes from your machine to this notebook — Colab runtimes have no inbound
address and no persistent URL, so a push is not available at any price. The loop is
therefore a **pull at the start of the run**: approve whenever you like, and the
next Colab run picks up everything approved since the last one.

Community rows are pinned to the **train** split. Validation and test stay pure
BTXRD, so scores remain comparable to every number in `overview.md`.

**The OOD manifest still has no learned consumer.** `onnm.ood` stage 1 is four
hand-tuned thresholds with no trainable component, so those negatives are measured
against rather than trained on — the cell reports the fraction the current
heuristics catch, which is the bar a learned gate would have to beat.

In [ ]:
# --- Community batch: approvals -> training data ----------------------------
# Needs your ADMIN key. Entered here rather than stored in Drive so it is not
# persisted alongside the dataset. Leave the URL blank to skip.
#
# The key alone is not enough: /admin/* also requires the request to name the one
# account permitted to review, which src/community.py sends automatically.
import json
import os
from getpass import getpass
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/OSTEONEURALNETWORK")
STORE = DRIVE_ROOT / "community"      # survives the runtime being wiped

os.environ["ONNM_COMMUNITY_URL"] = ""   # https://onnm-community.<sub>.workers.dev

if os.environ["ONNM_COMMUNITY_URL"]:
    os.environ["ONNM_ADMIN_KEY"] = getpass("ONNM_ADMIN_KEY (hidden): ")
    STORE.mkdir(parents=True, exist_ok=True)

    # Claims what you approved, writes the images into the Drive store, and
    # rebuilds configs/controls_manifest.csv -- which base.yaml already reads.
    # Nothing below needs editing as a result.
    !python scripts/sync_community.py --store "{STORE}" --note "colab $(date -u +%Y%m%d)"
elif STORE.exists():
    # No key to hand, but previous generations are already in the store: rebuild
    # from them so this run still trains on everything approved so far.
    !python scripts/sync_community.py --store "{STORE}" --rebuild-only
else:
    print("No community URL and no store - skipping. Training on BTXRD alone.")

# What will training actually see? Read the index rather than trusting the log.
index = STORE / "store.json"
if index.is_file():
    store = json.loads(index.read_text(encoding="utf-8"))
    print(f"\ncommunity store: {len(store['batches'])} batch(es), "
          f"{store['claimed_this_run']} claimed just now")
    print(f"  lesion rows in training: {store['lesion_rows']} {store['class_balance']}")
    print(f"  confirmed non-radiographs: {store['ood_rows']}")

    # Score the *current* gate against the confirmed negatives. This trains
    # nothing -- onnm.ood is heuristics with no learned component -- but it turns
    # the misc bucket into a measurement: the share of real, user-supplied misuse
    # stage 1 already rejects, and so the bar a learned gate would have to clear.
    #
    # It is a LOWER BOUND. Shared images are stored as single-channel PNGs, so the
    # colorfulness check -- the one that catches a photograph fastest -- cannot
    # fire on the stored copy the way it did on the upload. A miss here is not
    # proof the gate missed it live.
    ood_manifest = Path(store["ood_manifest"])
    if store["ood_rows"] and ood_manifest.is_file():
        import csv

        from onnm.ood import validate_payload

        with open(ood_manifest, encoding="utf-8") as handle:
            rows = list(csv.DictReader(handle))
        caught = sum(
            0 if validate_payload(Path(row["image"]).read_bytes(),
                                  row["image"]).is_radiograph else 1
            for row in rows
        )
        print(f"  stage-1 heuristics catch {caught}/{len(rows)} of them "
              f"(lower bound: stored copies are greyscale)")

## The two ablations

Each runs `overnight.yaml` with exactly one of its two changes removed. Together with the
existing `full` and `overnight` runs, that is enough to attribute the regression.

Budget roughly **45-70 min each** on a T4 at 40 epochs with early stopping. Run them one at
a time and keep the tab alive.


In [ ]:
# --- Cell 9: ablation A - OHEM alone ----------------------------------------
# Aggressive augmentation reverted to full_run strength; OHEM left on.
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --override configs/overnight.yaml \
    --override configs/ablations/ohem_only.yaml \
    --profile colab --epochs 40 --tag abl-ohem


In [ ]:
# --- Cell 10: ablation B - aggressive augmentation alone --------------------
# Augmentation as in overnight.yaml; OHEM disabled.
!python scripts/train.py \
    --override configs/densenet121_3class.yaml \
    --override configs/overnight.yaml \
    --override configs/ablations/augs_only.yaml \
    --profile colab --epochs 40 --tag abl-augs


In [ ]:
# --- Cell 11: calibrate and evaluate each run -------------------------------
# Threshold and temperature are fitted on VAL and applied unchanged to TEST.
# Never fit them on test (invariant 1) - scripts/calibrate.py warns if you try.
import glob

runs = sorted(glob.glob("reports/abl-*/best.pt"))
print("found:", runs)

for ckpt in runs:
    print("\n" + "=" * 70, "\n", ckpt, "\n" + "=" * 70)
    !python scripts/calibrate.py --checkpoint {ckpt} --profile colab --sweep
    !python scripts/evaluate.py  --checkpoint {ckpt} --profile colab --split test


In [ ]:
# --- Cell 12: side-by-side comparison ---------------------------------------
# The number that settles the question is val macro ROC-AUC, because it is
# threshold-independent: full 0.8905 vs overnight 0.8629. Whichever ablation
# lands near 0.86 is carrying the regression.
import glob
import json

rows = []
for hist_path in sorted(glob.glob("reports/*/history.json")):
    run = hist_path.split("/")[-2]
    with open(hist_path) as handle:
        history = json.load(handle)
    if not history:
        continue
    best = max(history, key=lambda e: e.get("val_roc_auc_macro", -1))
    rows.append({
        "run": run,
        "epochs": len(history),
        "val_auc": round(best.get("val_roc_auc_macro", float("nan")), 4),
        "val_mal_recall": round(best.get("val_malignant_recall", float("nan")), 3),
        "val_normal_called_lesion": best.get("val_normal_called_lesion"),
    })

try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values("val_auc", ascending=False))
except Exception:
    for r in rows:
        print(r)

print("\nreference (local runs):  full = 0.8905    overnight = 0.8629")


In [ ]:
# --- Cell 13: save results back to Drive ------------------------------------
# /content is wiped on disconnect. Copy the whole run directory, not just the
# checkpoint: history.json and metrics_*.json are what the comparison needs.
import shutil
from pathlib import Path

out = DRIVE / "reports"
out.mkdir(parents=True, exist_ok=True)

for run in sorted(Path("reports").glob("*")):
    if not run.is_dir():
        continue
    target = out / run.name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(run, target)
    print("saved", target)

print("\nDone. These persist in Drive after the runtime disconnects.")


## Licence

BTXRD is **CC BY-NC-ND 4.0**. Keeping a private copy in your own Drive is not
redistribution and is fine. What is not fine:

- publishing the dataset or any part of it
- publishing **Grad-CAM overlays** - they are derived images, which the NoDerivatives
  clause covers

`data/` and `reports/` are gitignored for this reason. Keep it that way.

## What to do with the result

Whichever ablation lands near **0.86** is the one carrying the regression:

- **`abl-ohem` low** -> OHEM is the cause. Lower `loss.ohem.penalty` below 4.0 or raise
  `warmup_epochs` above 5, and note that malignant recall fell 0.653 -> 0.469 while false
  positives fell 65 -> 37, which is a bias shift rather than better discrimination.
- **`abl-augs` low** -> the augmentation is too aggressive. `dropout_prob: 0.5` with up to
  6 holes of 40 px on a 256 px image can occlude a lesion outright, which teaches the model
  that a lesion-free-looking film is still labelled lesion.
- **Both near 0.89** -> the two interact, and the combination is the problem rather than
  either part.
